# Gene Ontology and Pathway Analysis

---

## Learning Objectives

By the end of this notebook you will be able to:

- Understand the Gene Ontology (GO) system: three ontologies, DAG structure, GO terms
- Perform GO enrichment analysis using the hypergeometric and Fisher's exact tests
- Use GOATOOLS and the g:Profiler concept for functional enrichment
- Navigate the KEGG database: structure, pathway maps, identifiers
- Access KEGG programmatically through its REST API
- Understand Gene Set Enrichment Analysis (GSEA)
- Predict transmembrane protein topology from sequence
- Build a complete workflow from a gene list to biological interpretation

---

[← Previous: Sequence Motifs and Protein Domains](../../Tier_2_Core_Bioinformatics/10_Sequence_Motifs_and_Domains/01_sequence_motifs_and_domains.ipynb) | [Next: Comparative Genomics →](../../Tier_2_Core_Bioinformatics/12_Comparative_Genomics/01_comparative_genomics.ipynb)

---

---

## 1. Gene Ontology (GO): Overview

The **Gene Ontology** is a controlled, structured vocabulary that describes gene and protein function in a species-independent way. It was created in 1998 to unify functional annotation across model organism databases.

GO has three independent **ontologies** (domains):

```
+---------------------------------------------------------------------------+
|                        GENE ONTOLOGY (GO)                                 |
+---------------------------------------------------------------------------+
|                                                                           |
|  1. MOLECULAR FUNCTION (MF)                                               |
|     What does the gene product DO at the molecular level?                 |
|     Examples: kinase activity, DNA binding, transporter activity           |
|                                                                           |
|  2. BIOLOGICAL PROCESS (BP)                                               |
|     What biological objective does it contribute to?                      |
|     Examples: cell cycle, apoptosis, signal transduction                  |
|                                                                           |
|  3. CELLULAR COMPONENT (CC)                                               |
|     WHERE in the cell does it function?                                   |
|     Examples: nucleus, mitochondrion, plasma membrane                     |
|                                                                           |
+---------------------------------------------------------------------------+
```

Each GO term has:
- A unique **GO ID** (e.g., `GO:0007049`)
- A **name** (e.g., "cell cycle")
- A **definition** (a textual description)
- One or more **relationships** to other terms (is_a, part_of, regulates)

---

### 1.1 The Directed Acyclic Graph (DAG)

GO terms are organized in a **Directed Acyclic Graph** -- not a simple tree. A child term can have multiple parents, allowing rich relationships.

```
biological_process (GO:0008150)
     |
     +-- cellular process (GO:0009987)
          |
          +-- cell cycle (GO:0007049)
          |    |
          |    +-- mitotic cell cycle (GO:0000278)
          |    |    |
          |    |    +-- G1/S transition (GO:0000082)
          |    |
          |    +-- meiotic cell cycle (GO:0051321)
          |
          +-- cell death (GO:0008219)
               |
               +-- apoptotic process (GO:0006915)
                    |
                    +-- intrinsic apoptotic signaling (GO:0097193)
```

**True path rule**: if a gene is annotated to a specific term, it is implicitly annotated to *all ancestor terms* up to the root. For example, a gene annotated to "G1/S transition" is also annotated to "mitotic cell cycle", "cell cycle", "cellular process", and "biological_process".

In [ ]:
# Build a mini Gene Ontology DAG

GO_TERMS = {
    # Molecular Function
    'GO:0003674': {'name': 'molecular_function', 'domain': 'MF', 'parents': []},
    'GO:0003824': {'name': 'catalytic activity', 'domain': 'MF', 'parents': ['GO:0003674']},
    'GO:0016772': {'name': 'transferase activity, transferring phosphorus-containing groups',
                   'domain': 'MF', 'parents': ['GO:0003824']},
    'GO:0016301': {'name': 'kinase activity', 'domain': 'MF', 'parents': ['GO:0016772']},
    'GO:0004672': {'name': 'protein kinase activity', 'domain': 'MF', 'parents': ['GO:0016301']},
    'GO:0005488': {'name': 'binding', 'domain': 'MF', 'parents': ['GO:0003674']},
    'GO:0003677': {'name': 'DNA binding', 'domain': 'MF', 'parents': ['GO:0005488']},

    # Biological Process
    'GO:0008150': {'name': 'biological_process', 'domain': 'BP', 'parents': []},
    'GO:0009987': {'name': 'cellular process', 'domain': 'BP', 'parents': ['GO:0008150']},
    'GO:0007049': {'name': 'cell cycle', 'domain': 'BP', 'parents': ['GO:0009987']},
    'GO:0000278': {'name': 'mitotic cell cycle', 'domain': 'BP', 'parents': ['GO:0007049']},
    'GO:0000082': {'name': 'G1/S transition of mitotic cell cycle', 'domain': 'BP',
                   'parents': ['GO:0000278']},
    'GO:0051321': {'name': 'meiotic cell cycle', 'domain': 'BP', 'parents': ['GO:0007049']},
    'GO:0008219': {'name': 'cell death', 'domain': 'BP', 'parents': ['GO:0009987']},
    'GO:0006915': {'name': 'apoptotic process', 'domain': 'BP', 'parents': ['GO:0008219']},
    'GO:0097193': {'name': 'intrinsic apoptotic signaling pathway', 'domain': 'BP',
                   'parents': ['GO:0006915']},
    'GO:0006260': {'name': 'DNA replication', 'domain': 'BP', 'parents': ['GO:0009987']},

    # Cellular Component
    'GO:0005575': {'name': 'cellular_component', 'domain': 'CC', 'parents': []},
    'GO:0005622': {'name': 'intracellular anatomical structure', 'domain': 'CC',
                   'parents': ['GO:0005575']},
    'GO:0005634': {'name': 'nucleus', 'domain': 'CC', 'parents': ['GO:0005622']},
    'GO:0005737': {'name': 'cytoplasm', 'domain': 'CC', 'parents': ['GO:0005622']},
    'GO:0005739': {'name': 'mitochondrion', 'domain': 'CC', 'parents': ['GO:0005737']},
    'GO:0005886': {'name': 'plasma membrane', 'domain': 'CC', 'parents': ['GO:0005575']},
}


def get_ancestors(go_id, go_terms):
    """Return all ancestor GO IDs for a given term (true path rule)."""
    ancestors = set()
    stack = list(go_terms.get(go_id, {}).get('parents', []))
    while stack:
        parent = stack.pop()
        if parent not in ancestors:
            ancestors.add(parent)
            stack.extend(go_terms.get(parent, {}).get('parents', []))
    return ancestors


# Demonstrate: ancestors of "G1/S transition"
term = 'GO:0000082'
ancestors = get_ancestors(term, GO_TERMS)
print(f"Term: {term} -- {GO_TERMS[term]['name']}")
print(f"\nAll ancestors (via true path rule):")
for a in sorted(ancestors):
    print(f"  {a}: {GO_TERMS[a]['name']}")

---

### 1.2 GO Annotations and Evidence Codes

A **GO annotation** links a gene product to a GO term, supported by an **evidence code** indicating how the annotation was determined.

```
+----------+----------------------------------------------+----------+
| Code     | Meaning                                      | Quality  |
+----------+----------------------------------------------+----------+
| Experimental Evidence                                               |
+----------+----------------------------------------------+----------+
| EXP      | Inferred from Experiment                     | High     |
| IDA      | Inferred from Direct Assay                   | High     |
| IPI      | Inferred from Physical Interaction           | High     |
| IMP      | Inferred from Mutant Phenotype               | High     |
| IGI      | Inferred from Genetic Interaction            | High     |
| IEP      | Inferred from Expression Pattern             | High     |
+----------+----------------------------------------------+----------+
| Computational Evidence                                              |
+----------+----------------------------------------------+----------+
| ISS      | Inferred from Sequence Similarity            | Medium   |
| ISO      | Inferred from Sequence Orthology             | Medium   |
| ISA      | Inferred from Sequence Alignment             | Medium   |
| IBA      | Inferred from Biological Aspect of Ancestor  | Medium   |
+----------+----------------------------------------------+----------+
| Author / Curator Evidence                                           |
+----------+----------------------------------------------+----------+
| TAS      | Traceable Author Statement                   | Medium   |
| NAS      | Non-traceable Author Statement               | Low      |
+----------+----------------------------------------------+----------+
| Automatic                                                           |
+----------+----------------------------------------------+----------+
| IEA      | Inferred from Electronic Annotation          | Low      |
+----------+----------------------------------------------+----------+
```

When filtering annotations, experimental evidence (EXP, IDA, IMP, ...) is the gold standard. IEA annotations are the most abundant but least reliable.

In [ ]:
# Gene-to-GO annotations with evidence codes (simplified real-world data)

GENE_ANNOTATIONS = {
    'TP53':  [('GO:0006915', 'IDA'), ('GO:0007049', 'IMP'), ('GO:0097193', 'TAS'),
             ('GO:0003677', 'IDA'), ('GO:0005634', 'IDA')],
    'CDK2':  [('GO:0004672', 'IDA'), ('GO:0000278', 'IDA'), ('GO:0000082', 'IMP'),
             ('GO:0005634', 'IDA'), ('GO:0005737', 'IEA')],
    'BCL2':  [('GO:0006915', 'IDA'), ('GO:0097193', 'IMP'), ('GO:0005739', 'IDA')],
    'CASP3': [('GO:0006915', 'IDA'), ('GO:0003824', 'IDA'), ('GO:0005737', 'IDA')],
    'CCNB1': [('GO:0000278', 'IDA'), ('GO:0005634', 'IDA'), ('GO:0005737', 'IDA')],
    'CCND1': [('GO:0000082', 'IDA'), ('GO:0000278', 'TAS'), ('GO:0005634', 'IDA')],
    'E2F1':  [('GO:0007049', 'IMP'), ('GO:0003677', 'IDA'), ('GO:0006260', 'TAS'),
             ('GO:0005634', 'IDA')],
    'RB1':   [('GO:0007049', 'IDA'), ('GO:0000082', 'IMP'), ('GO:0005634', 'IDA')],
    'CDKN1A':[('GO:0007049', 'IDA'), ('GO:0000082', 'IMP'), ('GO:0005634', 'IDA'),
             ('GO:0005737', 'IEA')],
    'BAX':   [('GO:0006915', 'IDA'), ('GO:0097193', 'IDA'), ('GO:0005739', 'IDA')],
    'MDM2':  [('GO:0006915', 'IMP'), ('GO:0005634', 'IDA'), ('GO:0005737', 'IDA')],
    'CASP9': [('GO:0006915', 'IDA'), ('GO:0097193', 'IDA'), ('GO:0003824', 'IDA'),
             ('GO:0005737', 'IDA')],
}

EVIDENCE_QUALITY = {
    'EXP': 5, 'IDA': 5, 'IPI': 5, 'IMP': 5, 'IGI': 5, 'IEP': 4,
    'ISS': 3, 'ISO': 3, 'ISA': 3, 'ISM': 3, 'IBA': 3,
    'TAS': 2, 'NAS': 1,
    'IEA': 1,
}


def filter_annotations_by_quality(gene_annotations, min_quality=3):
    """Keep only annotations supported by evidence at or above min_quality."""
    filtered = {}
    for gene, annots in gene_annotations.items():
        kept = [(go_id, ec) for go_id, ec in annots if EVIDENCE_QUALITY.get(ec, 0) >= min_quality]
        if kept:
            filtered[gene] = kept
    return filtered


# Show annotations for TP53
print("TP53 annotations (all):")
for go_id, ec in GENE_ANNOTATIONS['TP53']:
    term = GO_TERMS.get(go_id, {})
    print(f"  {go_id} {term.get('name', '?'):45s} [{term.get('domain', '?')}]  evidence={ec} (quality {EVIDENCE_QUALITY[ec]})")

# Filter to experimental evidence only
high_quality = filter_annotations_by_quality(GENE_ANNOTATIONS, min_quality=4)
print(f"\nGenes with high-quality annotations: {len(high_quality)}")
for gene in sorted(high_quality):
    terms = [go_id for go_id, _ in high_quality[gene]]
    print(f"  {gene}: {len(terms)} annotations")

---

## 2. GO Enrichment Analysis

Given a list of interesting genes (e.g., differentially expressed genes from an RNA-seq experiment), we ask: **are any GO terms over-represented in this list compared to the background?**

### 2.1 The Hypergeometric Test

The statistical model is the **hypergeometric distribution** (equivalent to Fisher's exact test for a 2x2 contingency table).

```
+--------------------+--------------+--------------+--------+
|                    | In GO term   | Not in term  | Total  |
+--------------------+--------------+--------------+--------+
| In gene list       |     k        |    n - k     |   n    |
| Not in gene list   |    K - k     |  N-K-n+k     |  N - n |
+--------------------+--------------+--------------+--------+
| Total              |     K        |    N - K     |   N    |
+--------------------+--------------+--------------+--------+

  N = total genes in background (e.g., genome)
  K = genes annotated to this GO term in the background
  n = genes in your list
  k = genes in your list AND annotated to this GO term

  P(X >= k) = sum_{i=k}^{min(n,K)} C(K,i)*C(N-K,n-i) / C(N,n)
```

A small p-value means the overlap `k` is larger than expected by chance -- the term is **enriched**.

In [ ]:
from scipy import stats
import numpy as np


def propagate_annotations(gene_annotations, go_terms):
    """
    Apply the true path rule: propagate each annotation to all ancestor terms.
    Returns dict  gene -> set of GO IDs (direct + propagated).
    """
    propagated = {}
    for gene, annots in gene_annotations.items():
        all_terms = set()
        for go_id, _ec in annots:
            all_terms.add(go_id)
            all_terms |= get_ancestors(go_id, go_terms)
        propagated[gene] = all_terms
    return propagated


def go_enrichment(gene_list, gene_annotations, go_terms, background_size=20000):
    """
    Perform GO enrichment analysis with the hypergeometric test.

    Parameters
    ----------
    gene_list : list of str
        Genes of interest (e.g., differentially expressed genes).
    gene_annotations : dict
        Gene -> list of (GO_ID, evidence_code).
    go_terms : dict
        GO_ID -> {'name', 'domain', 'parents'}.
    background_size : int
        Total number of genes in the background genome.

    Returns
    -------
    list of dict, sorted by p-value.
    """
    # Step 1: propagate annotations
    propagated = propagate_annotations(gene_annotations, go_terms)

    # Step 2: build term -> gene set mapping
    term_to_genes = {}
    for gene, terms in propagated.items():
        for t in terms:
            term_to_genes.setdefault(t, set()).add(gene)

    gene_set = set(g.upper() for g in gene_list)
    n = len(gene_set)
    N = background_size

    results = []
    for term_id, term_genes in term_to_genes.items():
        # Scale K for the full background (our annotation DB is small)
        annotated_in_list = gene_set & {g.upper() for g in term_genes}
        k = len(annotated_in_list)
        if k == 0:
            continue

        # Estimate K proportionally (in real analysis, use the full annotation DB)
        K = max(k, int(len(term_genes) / len(propagated) * N)) if propagated else k

        # Hypergeometric survival function: P(X >= k)
        p_value = stats.hypergeom.sf(k - 1, N, K, n)

        expected = n * K / N
        fold_enrichment = k / expected if expected > 0 else float('inf')

        info = go_terms.get(term_id, {})
        results.append({
            'go_id': term_id,
            'name': info.get('name', 'unknown'),
            'domain': info.get('domain', '?'),
            'k': k,
            'K': K,
            'p_value': p_value,
            'fold_enrichment': fold_enrichment,
            'genes': sorted(annotated_in_list),
        })

    results.sort(key=lambda r: r['p_value'])

    # Step 3: Benjamini-Hochberg FDR correction
    m = len(results)
    for i, r in enumerate(results):
        r['fdr'] = min(r['p_value'] * m / (i + 1), 1.0)

    return results

In [ ]:
# Run GO enrichment on an example gene list
# Scenario: RNA-seq identified these genes as differentially expressed in a cancer study

de_genes = ['TP53', 'BAX', 'CASP3', 'CASP9', 'BCL2', 'CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1']

enrichment_results = go_enrichment(de_genes, GENE_ANNOTATIONS, GO_TERMS, background_size=20000)

print("GO Enrichment Results")
print("=" * 85)
print(f"{'GO ID':<14} {'Domain':<4} {'Name':<42} {'k':>3} {'p-value':>10} {'FDR':>10}")
print("-" * 85)
for r in enrichment_results[:12]:
    print(f"{r['go_id']:<14} {r['domain']:<4} {r['name'][:40]:<42} {r['k']:>3} {r['p_value']:>10.2e} {r['fdr']:>10.2e}")
    print(f"{'':>14} Genes: {', '.join(r['genes'])}")

---

### 2.2 Multiple Testing Correction

When testing thousands of GO terms simultaneously, many will appear significant by chance alone. We must correct for **multiple testing**.

Common methods:

| Method | Description | Stringency |
|--------|-------------|------------|
| Bonferroni | Multiply p by number of tests | Very strict |
| Benjamini-Hochberg (BH) | Controls False Discovery Rate (FDR) | Moderate |
| Benjamini-Yekutieli | FDR for dependent tests | Moderate-strict |

In practice, **BH FDR < 0.05** is the standard threshold for GO enrichment.

In [ ]:
def multiple_testing_correction(p_values, method='bh'):
    """
    Apply multiple testing correction.

    Parameters
    ----------
    p_values : list of float
    method : str
        'bonferroni' or 'bh' (Benjamini-Hochberg)

    Returns
    -------
    list of float  (adjusted p-values)
    """
    m = len(p_values)
    if method == 'bonferroni':
        return [min(p * m, 1.0) for p in p_values]

    elif method == 'bh':
        # Sort p-values, apply BH, then restore original order
        indexed = sorted(enumerate(p_values), key=lambda x: x[1])
        adjusted = [0.0] * m
        prev_adj = 0.0
        for rank, (orig_idx, p) in enumerate(indexed, 1):
            adj = min(p * m / rank, 1.0)
            adjusted[orig_idx] = adj
        # Enforce monotonicity (larger rank => larger adjusted p)
        running_min = 1.0
        for rank in range(m, 0, -1):
            orig_idx = indexed[rank - 1][0]
            running_min = min(running_min, adjusted[orig_idx])
            adjusted[orig_idx] = running_min
        return adjusted

    raise ValueError(f"Unknown method: {method}")


# Demonstrate on example p-values
raw_p = [0.001, 0.03, 0.0001, 0.5, 0.04, 0.0005]
bonf = multiple_testing_correction(raw_p, method='bonferroni')
bh = multiple_testing_correction(raw_p, method='bh')

print(f"{'Raw p':>10} {'Bonferroni':>12} {'BH FDR':>10}")
print("-" * 35)
for p, b, f in zip(raw_p, bonf, bh):
    sig_raw = '*' if p < 0.05 else ' '
    sig_bh = '*' if f < 0.05 else ' '
    print(f"{p:>10.4f}{sig_raw} {b:>10.4f}   {f:>10.4f}{sig_bh}")
print("\n* = significant at alpha=0.05")

---

### 2.3 Tools for GO Enrichment

In practice, you will use established tools rather than writing enrichment tests from scratch:

**GOATOOLS** (Python)
```python
# Example (requires installation: pip install goatools)
from goatools.obo_parser import GODag
from goatools.go_enrichment import GOEnrichmentStudy

obodag = GODag('go-basic.obo')  # download from geneontology.org
# ... set up study genes, background, associations
goe = GOEnrichmentStudy(background_genes, associations, obodag, methods=['fdr_bh'])
results = goe.run_study(study_genes)
```

**g:Profiler** (web + Python client)
```python
# pip install gprofiler-official
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)
result = gp.profile(organism='hsapiens', query=['TP53', 'BAX', 'CASP3', ...])
```

**DAVID** (web-based): https://david.ncifcrf.gov/ -- upload gene lists, get enrichment tables.

**clusterProfiler** (R/Bioconductor): widely used for publication-quality figures.

In [ ]:
# Simulating a g:Profiler-style result for teaching purposes
# In real use, you would call the API as shown above

gprofiler_mock_result = [
    {'source': 'GO:BP', 'term_id': 'GO:0006915', 'term_name': 'apoptotic process',
     'p_value': 1.2e-8, 'term_size': 1287, 'intersection_size': 5,
     'intersections': ['TP53', 'BAX', 'BCL2', 'CASP3', 'CASP9']},
    {'source': 'GO:BP', 'term_id': 'GO:0007049', 'term_name': 'cell cycle',
     'p_value': 3.5e-7, 'term_size': 1543, 'intersection_size': 5,
     'intersections': ['TP53', 'CDK2', 'CCNB1', 'RB1', 'E2F1']},
    {'source': 'GO:MF', 'term_id': 'GO:0004672', 'term_name': 'protein kinase activity',
     'p_value': 2.1e-4, 'term_size': 512, 'intersection_size': 2,
     'intersections': ['CDK2', 'CCNB1']},
    {'source': 'GO:CC', 'term_id': 'GO:0005634', 'term_name': 'nucleus',
     'p_value': 8.7e-3, 'term_size': 6214, 'intersection_size': 7,
     'intersections': ['TP53', 'CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1', 'MDM2']},
    {'source': 'KEGG', 'term_id': 'KEGG:04210', 'term_name': 'Apoptosis',
     'p_value': 5.3e-9, 'term_size': 136, 'intersection_size': 5,
     'intersections': ['TP53', 'BAX', 'BCL2', 'CASP3', 'CASP9']},
    {'source': 'KEGG', 'term_id': 'KEGG:04110', 'term_name': 'Cell cycle',
     'p_value': 1.8e-7, 'term_size': 124, 'intersection_size': 5,
     'intersections': ['CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1']},
]

print("g:Profiler-style enrichment results")
print("=" * 90)
print(f"{'Source':<8} {'Term ID':<14} {'Term Name':<30} {'p-value':>10} {'Hits':>5} {'Size':>6}")
print("-" * 90)
for r in gprofiler_mock_result:
    print(f"{r['source']:<8} {r['term_id']:<14} {r['term_name']:<30} "
          f"{r['p_value']:>10.1e} {r['intersection_size']:>5} {r['term_size']:>6}")
    print(f"         Genes: {', '.join(r['intersections'])}")

---

## 3. KEGG Pathways

The **Kyoto Encyclopedia of Genes and Genomes (KEGG)** is a collection of databases for biological interpretation of genomic and molecular data.

### 3.1 KEGG Database Structure

```
+--------------------------------------------------------------------------+
|                        KEGG DATABASE STRUCTURE                            |
+--------------------------------------------------------------------------+
|                                                                          |
|  KEGG PATHWAY    -- Metabolic and regulatory pathway maps                |
|                     e.g., hsa00010 Glycolysis, hsa04110 Cell cycle        |
|                                                                          |
|  KEGG GENES      -- Gene catalogs for complete genomes                   |
|                     e.g., hsa:3098 = HK1 (hexokinase 1, human)           |
|                                                                          |
|  KEGG ORTHOLOGY  -- Functional orthologs across species (KO)             |
|                     e.g., K00844 = HK (hexokinase)                       |
|                                                                          |
|  KEGG COMPOUND   -- Small molecules / metabolites                        |
|                     e.g., C00031 = D-Glucose                             |
|                                                                          |
|  KEGG ENZYME     -- Enzyme nomenclature (EC numbers)                     |
|                     e.g., EC 2.7.1.1 = Hexokinase                        |
|                                                                          |
|  KEGG DISEASE    -- Human disease entries                                |
|  KEGG DRUG       -- Drug information                                     |
|                                                                          |
|  Organism codes: hsa=human, mmu=mouse, sce=yeast, eco=E.coli            |
|                                                                          |
+--------------------------------------------------------------------------+
```

### 3.2 KEGG Pathway Categories

KEGG pathways are organized hierarchically:

```
1. METABOLISM
   +-- Carbohydrate metabolism
   |   +-- 00010 Glycolysis / Gluconeogenesis
   |   +-- 00020 Citrate cycle (TCA cycle)
   +-- Energy metabolism
   |   +-- 00190 Oxidative phosphorylation
   +-- Lipid metabolism
   |   +-- 00061 Fatty acid biosynthesis
   +-- Amino acid metabolism
   +-- Nucleotide metabolism

2. GENETIC INFORMATION PROCESSING
   +-- 03010 Ribosome
   +-- 03030 DNA replication

3. ENVIRONMENTAL INFORMATION PROCESSING
   +-- 04010 MAPK signaling pathway
   +-- 04310 Wnt signaling pathway

4. CELLULAR PROCESSES
   +-- 04110 Cell cycle
   +-- 04210 Apoptosis
   +-- 04140 Autophagy

5. ORGANISMAL SYSTEMS
   +-- 04610 Complement and coagulation cascades

6. HUMAN DISEASES
   +-- 05200 Pathways in cancer
```

In [ ]:
# KEGG pathway database (representative example)

KEGG_PATHWAYS = {
    'hsa00010': {
        'name': 'Glycolysis / Gluconeogenesis',
        'category': 'Metabolism',
        'subcategory': 'Carbohydrate metabolism',
        'genes': ['HK1', 'HK2', 'GCK', 'GPI', 'PFKL', 'PFKM', 'ALDOA',
                  'TPI1', 'GAPDH', 'PGK1', 'PGAM1', 'ENO1', 'PKM', 'LDHA'],
    },
    'hsa00020': {
        'name': 'Citrate cycle (TCA cycle)',
        'category': 'Metabolism',
        'subcategory': 'Carbohydrate metabolism',
        'genes': ['CS', 'ACO1', 'ACO2', 'IDH1', 'IDH2', 'OGDH',
                  'SUCLA2', 'SUCLG1', 'SDHA', 'FH', 'MDH1', 'MDH2'],
    },
    'hsa04110': {
        'name': 'Cell cycle',
        'category': 'Cellular Processes',
        'subcategory': 'Cell growth and death',
        'genes': ['CDK1', 'CDK2', 'CDK4', 'CDK6', 'CCNA1', 'CCNB1',
                  'CCND1', 'CCNE1', 'RB1', 'E2F1', 'TP53', 'CDKN1A'],
    },
    'hsa04210': {
        'name': 'Apoptosis',
        'category': 'Cellular Processes',
        'subcategory': 'Cell growth and death',
        'genes': ['BAX', 'BAK1', 'BCL2', 'BCL2L1', 'CASP3', 'CASP8',
                  'CASP9', 'CYCS', 'APAF1', 'FAS', 'FASLG', 'TP53'],
    },
    'hsa04010': {
        'name': 'MAPK signaling pathway',
        'category': 'Environmental Information Processing',
        'subcategory': 'Signal transduction',
        'genes': ['MAPK1', 'MAPK3', 'MAP2K1', 'MAP2K2', 'RAF1', 'BRAF',
                  'KRAS', 'HRAS', 'NRAS', 'TP53', 'JUN', 'FOS'],
    },
    'hsa05200': {
        'name': 'Pathways in cancer',
        'category': 'Human Diseases',
        'subcategory': 'Cancer: overview',
        'genes': ['TP53', 'RB1', 'KRAS', 'BRAF', 'APC', 'PTEN',
                  'CDK4', 'CDK6', 'CCND1', 'E2F1', 'BAX', 'BCL2',
                  'CASP3', 'CASP9', 'MDM2', 'CDKN1A'],
    },
}


def find_pathways_for_gene(gene_symbol, pathway_db):
    """Find all pathways containing a given gene."""
    gene_upper = gene_symbol.upper()
    results = []
    for pid, info in pathway_db.items():
        if gene_upper in [g.upper() for g in info['genes']]:
            results.append((pid, info['name'], info['category']))
    return results


def find_shared_genes(pathway_id1, pathway_id2, pathway_db):
    """Find genes shared between two pathways."""
    genes1 = set(g.upper() for g in pathway_db[pathway_id1]['genes'])
    genes2 = set(g.upper() for g in pathway_db[pathway_id2]['genes'])
    return genes1 & genes2


# Which pathways contain TP53?
gene = 'TP53'
pathways = find_pathways_for_gene(gene, KEGG_PATHWAYS)
print(f"Pathways containing {gene}:")
for pid, name, cat in pathways:
    print(f"  {pid}: {name} [{cat}]")

# Genes shared between Cell cycle and Apoptosis
shared = find_shared_genes('hsa04110', 'hsa04210', KEGG_PATHWAYS)
print(f"\nGenes shared between Cell cycle and Apoptosis: {shared}")

---

### 3.3 KEGG REST API

KEGG provides a free REST API for programmatic access. The base URL is `https://rest.kegg.jp/`.

Key operations:

| Operation | URL pattern | Example |
|-----------|------------|----------|
| List pathways | `/list/pathway/hsa` | All human pathways |
| Get pathway info | `/get/hsa04110` | Cell cycle details |
| Find genes | `/link/hsa/pathway:hsa04110` | Genes in cell cycle |
| Search | `/find/pathway/apoptosis` | Search by keyword |
| Get image | `/get/hsa04110/image` | Pathway map image |

In [ ]:
import urllib.request
import urllib.error


def kegg_api_request(operation, *args):
    """
    Send a request to the KEGG REST API.

    Parameters
    ----------
    operation : str
        One of: list, get, find, link, conv
    *args : str
        Additional path segments.

    Returns
    -------
    str or None  (response text, or None on failure)
    """
    url = 'https://rest.kegg.jp/' + '/'.join([operation] + list(args))
    try:
        with urllib.request.urlopen(url, timeout=15) as response:
            return response.read().decode('utf-8')
    except (urllib.error.URLError, urllib.error.HTTPError) as e:
        print(f"KEGG API error: {e}")
        return None


def parse_kegg_list(text):
    """Parse tab-separated KEGG list output into list of (id, description) tuples."""
    if text is None:
        return []
    entries = []
    for line in text.strip().split('\n'):
        if '\t' in line:
            parts = line.split('\t', 1)
            entries.append((parts[0].strip(), parts[1].strip()))
    return entries


# Example 1: Search for apoptosis-related pathways
print("Searching KEGG for 'apoptosis':")
result = kegg_api_request('find', 'pathway', 'apoptosis')
if result:
    for entry_id, desc in parse_kegg_list(result)[:5]:
        print(f"  {entry_id}: {desc}")
else:
    print("  (API unavailable -- this is expected in offline environments)")

print()

# Example 2: Get info for a specific pathway
print("Getting details for hsa04210 (Apoptosis):")
result = kegg_api_request('get', 'hsa04210')
if result:
    # Print first 15 lines
    for line in result.split('\n')[:15]:
        print(f"  {line}")
    print("  ...")
else:
    print("  (API unavailable -- using local data instead)")

In [ ]:
def kegg_get_pathway_genes(pathway_id):
    """
    Get all gene IDs in a KEGG pathway via the API.
    Falls back to local data if the API is unreachable.
    """
    organism = pathway_id[:3]  # e.g., 'hsa'
    result = kegg_api_request('link', organism, f'pathway:{pathway_id}')
    if result:
        genes = []
        for line in result.strip().split('\n'):
            parts = line.split('\t')
            if len(parts) == 2:
                genes.append(parts[1].strip())
        return genes

    # Fallback to local data
    if pathway_id in KEGG_PATHWAYS:
        return [f"{organism}:{g}" for g in KEGG_PATHWAYS[pathway_id]['genes']]
    return []


# Example: get genes in the Cell cycle pathway
genes = kegg_get_pathway_genes('hsa04110')
print(f"Genes in hsa04110 (Cell cycle): {len(genes)} genes")
for g in genes[:10]:
    print(f"  {g}")
if len(genes) > 10:
    print(f"  ... and {len(genes) - 10} more")

---

## 4. Pathway Enrichment Analysis

Pathway enrichment uses the same statistical framework as GO enrichment (hypergeometric test), applied to curated pathway gene sets (KEGG, Reactome, WikiPathways).

In [ ]:
def pathway_enrichment(gene_list, pathway_db, background_size=20000):
    """
    Perform pathway enrichment analysis using the hypergeometric test.

    Parameters
    ----------
    gene_list : list of str
        Genes of interest.
    pathway_db : dict
        pathway_id -> {'name': str, 'genes': list}.
    background_size : int
        Total genes in the background genome.

    Returns
    -------
    list of dict, sorted by p-value.
    """
    gene_set = set(g.upper() for g in gene_list)
    n = len(gene_set)
    N = background_size

    results = []
    for pathway_id, info in pathway_db.items():
        pathway_genes = set(g.upper() for g in info['genes'])
        K = len(pathway_genes)

        overlap = gene_set & pathway_genes
        k = len(overlap)
        if k == 0:
            continue

        p_value = stats.hypergeom.sf(k - 1, N, K, n)
        expected = n * K / N
        fold = k / expected if expected > 0 else float('inf')

        results.append({
            'pathway_id': pathway_id,
            'name': info['name'],
            'category': info.get('category', ''),
            'p_value': p_value,
            'fold_enrichment': fold,
            'k': k,
            'K': K,
            'genes': sorted(overlap),
        })

    results.sort(key=lambda r: r['p_value'])

    # BH FDR correction
    m = len(results)
    for i, r in enumerate(results):
        r['fdr'] = min(r['p_value'] * m / (i + 1), 1.0)

    return results


# Scenario: genes from a cancer gene expression study
cancer_genes = [
    'TP53', 'BAX', 'BCL2', 'CASP3', 'CASP9',     # apoptosis
    'CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1',     # cell cycle
    'HK1', 'HK2', 'PFKL', 'GAPDH', 'ENO1', 'PKM', # glycolysis (Warburg effect)
    'KRAS', 'BRAF', 'MDM2',                         # oncogenes
]

results = pathway_enrichment(cancer_genes, KEGG_PATHWAYS)

print("KEGG Pathway Enrichment Results")
print("=" * 85)
for r in results:
    sig = '***' if r['fdr'] < 0.001 else '**' if r['fdr'] < 0.01 else '*' if r['fdr'] < 0.05 else ''
    print(f"\n{r['pathway_id']}: {r['name']} [{r['category']}] {sig}")
    print(f"  p-value: {r['p_value']:.2e}   FDR: {r['fdr']:.2e}   Fold enrichment: {r['fold_enrichment']:.1f}x")
    print(f"  Overlap: {r['k']}/{r['K']} pathway genes   Genes: {', '.join(r['genes'])}")

---

### 4.1 Gene Set Enrichment Analysis (GSEA)

Standard enrichment (Over-Representation Analysis, ORA) requires a **predefined gene list** -- you must choose a threshold to define "interesting" genes. GSEA avoids this limitation.

```
+--------------------------------------------------------------------------+
|                  ORA vs. GSEA                                             |
+--------------------------------------------------------------------------+
|                                                                          |
|  ORA (Over-Representation Analysis):                                     |
|    1. Select DE genes (e.g., padj < 0.05, |logFC| > 1)                   |
|    2. Test: are pathway genes over-represented in this list?              |
|    Problem: arbitrary threshold; ignores magnitude of change              |
|                                                                          |
|  GSEA (Gene Set Enrichment Analysis):                                    |
|    1. Rank ALL genes by a metric (e.g., log2 fold-change)                |
|    2. Walk down the ranked list; for each gene in the pathway,           |
|       increase a running sum; for each gene NOT in the pathway,          |
|       decrease it.                                                       |
|    3. The Enrichment Score (ES) = maximum deviation from zero.           |
|    4. Assess significance by permutation testing.                        |
|                                                                          |
|  Ranked gene list:                                                       |
|  Gene:    A  B  C  D  E  F  G  H  I  J  K  L  M  N  O  P              |
|  logFC: +4 +3 +2 +2 +1 +1  0  0  0 -1 -1 -2 -2 -3 -3 -4              |
|  In set:  *        *  *           *              *                       |
|                    -----> ES tracks pathway gene positions               |
|                                                                          |
+--------------------------------------------------------------------------+
```

In [ ]:
def gsea_enrichment_score(ranked_genes, gene_set):
    """
    Calculate GSEA enrichment score for a gene set.

    Parameters
    ----------
    ranked_genes : list of (gene_name, rank_metric)
        All genes sorted by fold-change (descending).
    gene_set : set of str
        Genes in the pathway/term of interest.

    Returns
    -------
    dict with 'ES', 'running_sum', 'hits'
    """
    gene_set_upper = {g.upper() for g in gene_set}
    N = len(ranked_genes)
    Nh = sum(1 for g, _ in ranked_genes if g.upper() in gene_set_upper)
    Nm = N - Nh

    if Nh == 0 or Nm == 0:
        return {'ES': 0.0, 'running_sum': [], 'hits': []}

    # Weight by absolute rank metric
    Nr = sum(abs(metric) for g, metric in ranked_genes if g.upper() in gene_set_upper)

    running_sum = []
    hits = []
    current = 0.0

    for i, (gene, metric) in enumerate(ranked_genes):
        if gene.upper() in gene_set_upper:
            current += abs(metric) / Nr if Nr > 0 else 0
            hits.append(i)
        else:
            current -= 1.0 / Nm
        running_sum.append(current)

    # ES = maximum absolute deviation
    max_pos = max(running_sum)
    min_neg = min(running_sum)
    ES = max_pos if abs(max_pos) >= abs(min_neg) else min_neg

    return {'ES': ES, 'running_sum': running_sum, 'hits': hits}


# Simulate a ranked gene list (e.g., from RNA-seq differential expression)
np.random.seed(42)
all_genes_with_fc = [
    ('TP53', 3.2), ('BAX', 2.8), ('CASP3', 2.5), ('CASP9', 2.1),
    ('BCL2', -1.5), ('MAPK1', 1.1), ('GAPDH', 0.9), ('CDK2', 1.8),
    ('CCNB1', 1.5), ('HK1', 2.0), ('HK2', 1.7), ('ENO1', 1.3),
    ('CCND1', 0.8), ('RB1', -0.5), ('E2F1', 0.7), ('PFKL', 1.4),
    ('MDM2', -2.1), ('KRAS', 0.4), ('PKM', 1.2), ('FAS', -0.3),
]
# Add 80 "background" genes with small fold-changes
for i in range(80):
    all_genes_with_fc.append((f'GENE_{i}', np.random.normal(0, 0.5)))

# Sort by fold-change descending
all_genes_with_fc.sort(key=lambda x: x[1], reverse=True)

# Test apoptosis gene set
apoptosis_genes = {'BAX', 'BAK1', 'BCL2', 'CASP3', 'CASP8', 'CASP9', 'TP53', 'FAS', 'FASLG'}
result = gsea_enrichment_score(all_genes_with_fc, apoptosis_genes)

print(f"GSEA for Apoptosis gene set:")
print(f"  Enrichment Score (ES): {result['ES']:.3f}")
print(f"  Positive ES => gene set is enriched at the TOP of the ranked list")
print(f"  (i.e., apoptosis genes tend to be UP-regulated)")
print(f"\n  Hit positions in ranked list: {result['hits']}")
for h in result['hits']:
    gene, fc = all_genes_with_fc[h]
    print(f"    Rank {h:3d}: {gene:<10s} logFC = {fc:+.2f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_gsea(ranked_genes, gene_set, title='GSEA Enrichment Plot'):
    """Visualize GSEA running enrichment score."""
    result = gsea_enrichment_score(ranked_genes, gene_set)

    fig, axes = plt.subplots(3, 1, figsize=(10, 6), height_ratios=[3, 0.5, 1],
                             sharex=True, gridspec_kw={'hspace': 0.05})

    # Top panel: running enrichment score
    axes[0].plot(result['running_sum'], color='green', linewidth=2)
    axes[0].axhline(0, color='grey', linewidth=0.5)
    es_idx = np.argmax(np.abs(result['running_sum']))
    axes[0].axvline(es_idx, color='red', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('Enrichment Score')
    axes[0].set_title(title)
    axes[0].text(es_idx + 1, result['ES'], f'ES={result["ES"]:.2f}', color='red')

    # Middle panel: hit markers
    for h in result['hits']:
        axes[1].axvline(h, color='black', linewidth=0.8)
    axes[1].set_ylabel('Hits')
    axes[1].set_yticks([])

    # Bottom panel: rank metric
    metrics = [m for _, m in ranked_genes]
    axes[2].fill_between(range(len(metrics)), metrics, color='lightblue')
    axes[2].axhline(0, color='grey', linewidth=0.5)
    axes[2].set_ylabel('Rank metric')
    axes[2].set_xlabel('Gene rank')

    plt.tight_layout()
    plt.show()


plot_gsea(all_genes_with_fc, apoptosis_genes, title='GSEA: Apoptosis gene set')

---

## 5. Transmembrane Protein Prediction

Transmembrane (TM) proteins span the lipid bilayer with one or more hydrophobic alpha-helical segments of approximately 20-25 residues. Predicting TM topology from sequence is a classic bioinformatics task.

### 5.1 Membrane Protein Architecture

```
Extracellular
===========================================================================
    N-term          Loop1            Loop2
      |               |                |
------+---------+-----+--------+-------+--------+---------+------
      |  TM1    |              |  TM2   |                |  TM3  |
      | (helix) |              | (helix)|                |(helix)|
------+---------+--------------+--------+----------------+-------+------
                    |                        |                |
                  Loop (inside)           Loop (inside)     C-term
===========================================================================
Cytoplasm

TM helix characteristics:
  - 20-25 hydrophobic amino acids
  - alpha-helical (backbone H-bonds are internal)
  - Rich in Leu, Ile, Val, Ala, Phe
  - Spans ~30 Angstroms of membrane

Topology types:
  Type I    : N-out, C-in, single TM  (e.g., glycophorin)
  Type II   : N-in, C-out, single TM  (e.g., transferrin receptor)
  Multi-pass: Multiple TM helices      (e.g., GPCRs have 7 TM helices)
```

### 5.2 Kyte-Doolittle Hydropathy Plot

The classic method to predict TM regions uses a sliding-window average of amino acid hydropathy values. Regions with average hydropathy > 1.6 (window size 19) are predicted as TM.

In [ ]:
# Kyte-Doolittle hydropathy scale
HYDROPATHY = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
}


def hydropathy_profile(sequence, window_size=19):
    """
    Calculate hydropathy profile using a sliding window.

    Returns list of (position, average_hydropathy) tuples.
    """
    half = window_size // 2
    profile = []
    for i in range(half, len(sequence) - half):
        window = sequence[i - half:i + half + 1]
        avg = sum(HYDROPATHY.get(aa, 0) for aa in window) / window_size
        profile.append((i + 1, avg))  # 1-indexed position
    return profile


def predict_tm_regions(sequence, window_size=19, threshold=1.6, min_length=15):
    """
    Predict transmembrane regions from a hydropathy profile.

    Returns (tm_regions, profile) where tm_regions is a list of (start, end) tuples.
    """
    profile = hydropathy_profile(sequence, window_size)

    tm_regions = []
    in_tm = False
    tm_start = None

    for pos, val in profile:
        if val > threshold:
            if not in_tm:
                tm_start = pos
                in_tm = True
        else:
            if in_tm:
                if pos - tm_start >= min_length:
                    tm_regions.append((tm_start, pos - 1))
                in_tm = False

    if in_tm:
        end_pos = profile[-1][0]
        if end_pos - tm_start >= min_length:
            tm_regions.append((tm_start, end_pos))

    return tm_regions, profile


# Example: a synthetic multi-pass membrane protein
# (hydrophobic TM segments separated by hydrophilic loops)
example_protein = (
    "MEKSDQRYNP"          # N-terminus (extracellular)
    "LLIVFILAGVILLA"      # TM1 (14 aa)
    "VVIFAGLLLWILAM"      # TM1 continued (14 aa) -- total ~28 aa hydrophobic
    "ERKQTWDNSQP"         # Loop 1 (cytoplasmic)
    "AVVLAFLLWGLAMI"      # TM2
    "FLFVIGALLIVLFL"      # TM2 continued
    "DRSQWYNPKEKQT"       # Loop 2 (extracellular)
    "FFILCALLIIVFLA"      # TM3
    "GALIVFLLWVLFLV"      # TM3 continued
    "QSKTEVNDHRQQMP"      # C-terminus (cytoplasmic)
)

tm_regions, profile = predict_tm_regions(example_protein)

print(f"Sequence length: {len(example_protein)} aa")
print(f"\nPredicted transmembrane regions:")
for i, (start, end) in enumerate(tm_regions, 1):
    segment = example_protein[start - 1:end]
    print(f"  TM{i}: positions {start}-{end} ({end - start + 1} aa)")
    print(f"        {segment}")

In [ ]:
def plot_hydropathy(sequence, window_size=19, threshold=1.6):
    """Plot Kyte-Doolittle hydropathy profile with predicted TM regions."""
    tm_regions, profile = predict_tm_regions(sequence, window_size, threshold)
    positions = [p for p, _ in profile]
    values = [v for _, v in profile]

    fig, ax = plt.subplots(figsize=(12, 4))

    # Plot profile
    ax.plot(positions, values, color='steelblue', linewidth=1.5)
    ax.axhline(y=threshold, color='red', linestyle='--', alpha=0.7, label=f'Threshold ({threshold})')
    ax.axhline(y=0, color='grey', linewidth=0.5)

    # Shade TM regions
    for start, end in tm_regions:
        ax.axvspan(start, end, alpha=0.25, color='orange', label='Predicted TM')

    ax.set_xlabel('Residue position')
    ax.set_ylabel('Hydropathy score')
    ax.set_title(f'Kyte-Doolittle Hydropathy Plot (window={window_size})')

    # De-duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys())

    plt.tight_layout()
    plt.show()


plot_hydropathy(example_protein)

### 5.3 Topology Prediction: Positive-Inside Rule

Cytoplasmic loops tend to contain more positively charged residues (Arg, Lys) than extracellular loops. This **positive-inside rule** helps determine the orientation of a membrane protein.

In [ ]:
def predict_topology(sequence, tm_regions):
    """
    Predict membrane protein topology using the positive-inside rule.

    Returns dict with orientation, loop details, and TM count.
    """
    # Extract loops between TM segments (and terminal regions)
    loops = []
    prev_end = 0

    for start, end in tm_regions:
        if start > prev_end + 1:
            loop_seq = sequence[prev_end:start - 1]
            loops.append({'start': prev_end + 1, 'end': start - 1,
                          'sequence': loop_seq, 'length': len(loop_seq)})
        prev_end = end

    # C-terminal region
    if prev_end < len(sequence):
        loop_seq = sequence[prev_end:]
        loops.append({'start': prev_end + 1, 'end': len(sequence),
                      'sequence': loop_seq, 'length': len(loop_seq)})

    # Count Arg + Lys in each loop
    for loop in loops:
        loop['KR_count'] = sum(1 for aa in loop['sequence'] if aa in 'KR')
        loop['KR_density'] = loop['KR_count'] / loop['length'] if loop['length'] > 0 else 0

    # Assign inside/outside by alternating and applying positive-inside rule
    odd_kr = sum(loops[i]['KR_count'] for i in range(0, len(loops), 2))
    even_kr = sum(loops[i]['KR_count'] for i in range(1, len(loops), 2))

    if odd_kr >= even_kr:
        # N-terminus is on the cytoplasmic side (inside)
        orientation = 'N-in (Type II or multi-pass)'
        for i, loop in enumerate(loops):
            loop['location'] = 'cytoplasm' if i % 2 == 0 else 'extracellular'
    else:
        orientation = 'N-out (Type I or multi-pass)'
        for i, loop in enumerate(loops):
            loop['location'] = 'extracellular' if i % 2 == 0 else 'cytoplasm'

    return {'orientation': orientation, 'loops': loops, 'tm_count': len(tm_regions)}


topology = predict_topology(example_protein, tm_regions)

print("Topology Prediction")
print("=" * 55)
print(f"Predicted orientation: {topology['orientation']}")
print(f"Number of TM helices:  {topology['tm_count']}")
print(f"\nLoop analysis:")
for loop in topology['loops']:
    print(f"  Residues {loop['start']:>3}-{loop['end']:<3}  {loop['location']:<15}  "
          f"K+R={loop['KR_count']}  ({loop['length']} aa)")

### 5.4 TMHMM and Other Prediction Tools

In practice, prediction of transmembrane topology relies on machine-learning methods that outperform simple hydropathy:

| Tool | Method | URL |
|------|--------|-----|
| TMHMM 2.0 | Hidden Markov Model | services.healthtech.dtu.dk |
| Phobius | HMM (combined TM + signal peptide) | phobius.sbc.su.se |
| DeepTMHMM | Deep learning | biolib.com/DTU/DeepTMHMM |
| TOPCONS | Consensus of multiple methods | topcons.net |

**TMHMM** is the classic tool. It models the sequence as a Hidden Markov Model with states for:
- Inside (cytoplasm) loop
- Outside (extracellular) loop
- Transmembrane helix (with sub-states for helix cap and core)

The model was trained on experimentally characterized membrane proteins.

---

## 6. Putting It All Together: From Gene List to Biological Interpretation

Here is a complete workflow for functional annotation of a gene list:

```
Experiment (e.g., RNA-seq, proteomics)
         |
         v
   Gene list (e.g., DE genes)
         |
    +----+----+----+
    |         |    |
    v         v    v
  GO         KEGG  Reactome / WikiPathways
  enrichment pathway   enrichment
    |         |    |
    v         v    v
  Biological  Pathway    Visualization
  processes   maps       (bar charts, network plots)
    |         |    |
    +----+----+----+
         |
         v
   Biological interpretation
   - Which processes are affected?
   - Which pathways are activated/repressed?
   - Hypothesis generation for follow-up experiments
```

In [ ]:
def functional_annotation_workflow(gene_list, gene_annotations, go_terms,
                                    pathway_db, background_size=20000):
    """
    Complete functional annotation pipeline.

    Parameters
    ----------
    gene_list : list of str
    gene_annotations : dict
    go_terms : dict
    pathway_db : dict
    background_size : int

    Returns
    -------
    dict with 'go_results' and 'pathway_results'
    """
    print(f"Functional Annotation Workflow")
    print(f"Input: {len(gene_list)} genes")
    print(f"Background size: {background_size:,} genes")
    print("=" * 60)

    # Step 1: GO enrichment
    print("\n--- Step 1: GO Enrichment ---")
    go_results = go_enrichment(gene_list, gene_annotations, go_terms, background_size)
    sig_go = [r for r in go_results if r['fdr'] < 0.05]
    print(f"Tested GO terms: {len(go_results)}")
    print(f"Significant (FDR < 0.05): {len(sig_go)}")
    for r in sig_go[:5]:
        print(f"  {r['go_id']} {r['name'][:35]:<35s} [{r['domain']}] FDR={r['fdr']:.2e}")

    # Step 2: Pathway enrichment
    print("\n--- Step 2: KEGG Pathway Enrichment ---")
    pw_results = pathway_enrichment(gene_list, pathway_db, background_size)
    sig_pw = [r for r in pw_results if r['fdr'] < 0.05]
    print(f"Tested pathways: {len(pw_results)}")
    print(f"Significant (FDR < 0.05): {len(sig_pw)}")
    for r in sig_pw:
        print(f"  {r['pathway_id']} {r['name']:<35s} FDR={r['fdr']:.2e}  ({r['k']}/{r['K']} genes)")

    # Step 3: Summary
    print("\n--- Step 3: Summary ---")
    bp_terms = [r for r in sig_go if r['domain'] == 'BP']
    mf_terms = [r for r in sig_go if r['domain'] == 'MF']
    cc_terms = [r for r in sig_go if r['domain'] == 'CC']
    print(f"Significant GO-BP terms: {len(bp_terms)}")
    print(f"Significant GO-MF terms: {len(mf_terms)}")
    print(f"Significant GO-CC terms: {len(cc_terms)}")
    print(f"Significant KEGG pathways: {len(sig_pw)}")

    return {'go_results': go_results, 'pathway_results': pw_results}


# Run the full workflow
results = functional_annotation_workflow(
    cancer_genes, GENE_ANNOTATIONS, GO_TERMS, KEGG_PATHWAYS
)

In [ ]:
def plot_enrichment_barplot(results, title='Enrichment Results', max_terms=10):
    """Create a bar chart of enrichment results (negative log10 FDR)."""
    sig = [r for r in results if r.get('fdr', 1) < 1.0][:max_terms]
    if not sig:
        print("No significant results to plot.")
        return

    sig = sig[::-1]  # reverse so most significant is at top

    # Determine which key holds the term name
    name_key = 'name'
    labels = [f"{r.get('go_id', r.get('pathway_id', ''))}: {r[name_key][:30]}" for r in sig]
    neg_log_fdr = [-np.log10(r['fdr']) if r['fdr'] > 0 else 20 for r in sig]

    fig, ax = plt.subplots(figsize=(9, max(3, len(sig) * 0.4)))
    colors = ['steelblue' if 'go_id' in r else 'darkorange' for r in sig]
    ax.barh(range(len(labels)), neg_log_fdr, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel('-log10(FDR)')
    ax.axvline(-np.log10(0.05), color='red', linestyle='--', label='FDR = 0.05')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_enrichment_barplot(results['go_results'], title='GO Enrichment')
plot_enrichment_barplot(results['pathway_results'], title='KEGG Pathway Enrichment')

---

## 7. Exercises

### Exercise 1: Annotate a Gene List

Given the following gene list from a proteomics experiment, find all GO annotations for each gene using our `GENE_ANNOTATIONS` dictionary. For each gene, list the GO term names and domains.

In [ ]:
# Exercise 1
proteomics_genes = ['TP53', 'CDK2', 'CASP3', 'BCL2', 'E2F1', 'MDM2']

# YOUR CODE HERE: for each gene, print its GO annotations with term name and domain


In [ ]:
# Solution 1
proteomics_genes = ['TP53', 'CDK2', 'CASP3', 'BCL2', 'E2F1', 'MDM2']

for gene in proteomics_genes:
    annots = GENE_ANNOTATIONS.get(gene, [])
    print(f"\n{gene} ({len(annots)} annotations):")
    if not annots:
        print("  No annotations found.")
        continue
    for go_id, ec in annots:
        term = GO_TERMS.get(go_id, {})
        name = term.get('name', 'unknown')
        domain = term.get('domain', '?')
        print(f"  {go_id} {name:<45s} [{domain}]  evidence: {ec}")

### Exercise 2: GO Enrichment with Different Backgrounds

Run the `go_enrichment` function with three different background sizes (5,000; 20,000; 50,000) and compare the results. How does the background size affect p-values and fold enrichment?

In [ ]:
# Exercise 2
test_genes = ['TP53', 'BAX', 'CASP3', 'CASP9', 'BCL2']

# YOUR CODE HERE: run go_enrichment with background_size = 5000, 20000, 50000
# Compare the top result's p-value and fold_enrichment across the three runs


In [ ]:
# Solution 2
test_genes = ['TP53', 'BAX', 'CASP3', 'CASP9', 'BCL2']

for bg in [5000, 20000, 50000]:
    res = go_enrichment(test_genes, GENE_ANNOTATIONS, GO_TERMS, background_size=bg)
    top = res[0] if res else None
    if top:
        print(f"Background={bg:>6,}:  top term = {top['go_id']} ({top['name'][:25]})  "
              f"p={top['p_value']:.2e}  fold={top['fold_enrichment']:.1f}x")

print("\nConclusion: larger background -> smaller p-values (more significant)")
print("because the same overlap k becomes more surprising against a bigger background.")

### Exercise 3: KEGG Pathway Overlap Network

For each pair of pathways in `KEGG_PATHWAYS`, find the number of shared genes. Which pathway pair has the most overlap?

In [ ]:
# Exercise 3

# YOUR CODE HERE: compute pairwise gene overlaps between all pathways
# Print a matrix or a ranked list of pathway pairs by overlap size


In [ ]:
# Solution 3
from itertools import combinations

pathway_ids = sorted(KEGG_PATHWAYS.keys())

print(f"{'Pathway 1':<12} {'Pathway 2':<12} {'Shared':>6}  Genes")
print("-" * 65)

overlaps = []
for p1, p2 in combinations(pathway_ids, 2):
    shared = find_shared_genes(p1, p2, KEGG_PATHWAYS)
    if shared:
        overlaps.append((p1, p2, shared))

overlaps.sort(key=lambda x: len(x[2]), reverse=True)

for p1, p2, shared in overlaps:
    name1 = KEGG_PATHWAYS[p1]['name'][:20]
    name2 = KEGG_PATHWAYS[p2]['name'][:20]
    print(f"{p1} ({name1})  x  {p2} ({name2})  shared={len(shared)}")
    print(f"    Genes: {', '.join(sorted(shared))}")

print(f"\nMost overlapping pair: {overlaps[0][0]} and {overlaps[0][1]} "
      f"with {len(overlaps[0][2])} shared genes.")

### Exercise 4: Hydropathy Plot for a Real Protein

The sequence below is human **Aquaporin-1** (AQP1), a water channel with 6 transmembrane helices. Calculate its hydropathy profile and predict TM regions.

In [ ]:
# Exercise 4

# Human Aquaporin-1 (UniProt P29972), 269 residues
aqp1_sequence = (
    "MASEFKKKLFWRAVVAEFLAMTLVFISIGSALGFHYNNQTLAGKVLISLICQLAIAIF"
    "FATDSLDNLASLVDNPTPVDAAVRGKDYDAKWDETHTHPGDLIRTLALSGETFTSYP"
    "HGIMAVDISSGNLSQYGPAVDIGITAVGHVNMIFSSKGHISSAGAAIYNKIVTSGLS"
    "QLGSVALAILGFNIIQAGLGVCIELFKASASGSGHLNPAVTLGLLLSCQISIFRALL"
    "GNDAGAKASYTTSTVQLMVELKDQPI"
)

# YOUR CODE HERE:
# 1. Call predict_tm_regions on aqp1_sequence
# 2. Print the predicted TM regions
# 3. Call plot_hydropathy to visualize
# 4. Compare your prediction with the known 6 TM helices


In [ ]:
# Solution 4

aqp1_sequence = (
    "MASEFKKKLFWRAVVAEFLAMTLVFISIGSALGFHYNNQTLAGKVLISLICQLAIAIF"
    "FATDSLDNLASLVDNPTPVDAAVRGKDYDAKWDETHTHPGDLIRTLALSGETFTSYP"
    "HGIMAVDISSGNLSQYGPAVDIGITAVGHVNMIFSSKGHISSAGAAIYNKIVTSGLS"
    "QLGSVALAILGFNIIQAGLGVCIELFKASASGSGHLNPAVTLGLLLSCQISIFRALL"
    "GNDAGAKASYTTSTVQLMVELKDQPI"
)

tm_regions_aqp, profile_aqp = predict_tm_regions(aqp1_sequence, threshold=1.6, min_length=12)

print(f"Aquaporin-1: {len(aqp1_sequence)} residues")
print(f"Predicted TM regions: {len(tm_regions_aqp)}")
for i, (s, e) in enumerate(tm_regions_aqp, 1):
    print(f"  TM{i}: {s}-{e} ({e - s + 1} aa)  {aqp1_sequence[s-1:e]}")

print("\nNote: Aquaporins have an unusual topology with 6 TM helices and")
print("two re-entrant loops (HB and HE) that do not fully cross the membrane.")
print("Simple hydropathy may not perfectly capture all 6 TM segments.")

plot_hydropathy(aqp1_sequence, threshold=1.6)

### Exercise 5: Visualize Pathway Enrichment

Using the `pathway_enrichment` function, analyze the gene list below and create a horizontal bar chart showing all pathways, coloring significant ones (FDR < 0.05) differently from non-significant.

In [ ]:
# Exercise 5

my_genes = ['TP53', 'BAX', 'BCL2', 'CASP3', 'CASP9',  # apoptosis
            'CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1',  # cell cycle
            'KRAS', 'BRAF', 'MDM2']                     # cancer signaling

# YOUR CODE HERE:
# 1. Run pathway_enrichment
# 2. Create a bar chart with -log10(FDR) on x-axis, pathway names on y-axis
# 3. Color significant pathways differently from non-significant


In [ ]:
# Solution 5

my_genes = ['TP53', 'BAX', 'BCL2', 'CASP3', 'CASP9',
            'CDK2', 'CCNB1', 'CCND1', 'RB1', 'E2F1',
            'KRAS', 'BRAF', 'MDM2']

pw_res = pathway_enrichment(my_genes, KEGG_PATHWAYS)

# Print numerical results
print("Pathway enrichment results:")
print(f"{'Pathway':<40s} {'p-value':>10} {'FDR':>10} {'Genes':>5}")
print("-" * 70)
for r in pw_res:
    sig = '*' if r['fdr'] < 0.05 else ''
    print(f"{r['name']:<40s} {r['p_value']:>10.2e} {r['fdr']:>10.2e} {r['k']:>5}{sig}")

# Bar chart
pw_res_rev = pw_res[::-1]
labels = [r['name'] for r in pw_res_rev]
values = [-np.log10(r['fdr']) if r['fdr'] > 0 else 20 for r in pw_res_rev]
colors = ['#c0392b' if r['fdr'] < 0.05 else '#95a5a6' for r in pw_res_rev]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(range(len(labels)), values, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.axvline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8, label='FDR=0.05')
ax.set_xlabel('-log10(FDR)')
ax.set_title('KEGG Pathway Enrichment')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 6: Build a GSEA Running-Sum Plot

Using the provided `all_genes_with_fc` ranked list, compute and plot the GSEA enrichment score for the **Cell cycle** gene set. Is the cell cycle gene set enriched at the top or bottom of the list?

In [ ]:
# Exercise 6

cell_cycle_genes = {'CDK1', 'CDK2', 'CDK4', 'CDK6', 'CCNA1', 'CCNB1',
                    'CCND1', 'CCNE1', 'RB1', 'E2F1', 'TP53', 'CDKN1A'}

# YOUR CODE HERE:
# 1. Call gsea_enrichment_score(all_genes_with_fc, cell_cycle_genes)
# 2. Print the ES and interpret its sign
# 3. Call plot_gsea to visualize


In [ ]:
# Solution 6

cell_cycle_genes = {'CDK1', 'CDK2', 'CDK4', 'CDK6', 'CCNA1', 'CCNB1',
                    'CCND1', 'CCNE1', 'RB1', 'E2F1', 'TP53', 'CDKN1A'}

gsea_result = gsea_enrichment_score(all_genes_with_fc, cell_cycle_genes)
print(f"Cell cycle GSEA:")
print(f"  Enrichment Score: {gsea_result['ES']:.3f}")
if gsea_result['ES'] > 0:
    print("  Interpretation: cell cycle genes are enriched at the TOP of the ranked list")
    print("  -> cell cycle genes tend to be UP-regulated in this experiment")
else:
    print("  Interpretation: cell cycle genes are enriched at the BOTTOM of the ranked list")
    print("  -> cell cycle genes tend to be DOWN-regulated in this experiment")

plot_gsea(all_genes_with_fc, cell_cycle_genes, title='GSEA: Cell Cycle Gene Set')

---

## Summary

| Topic | Key Concept |
|-------|------------|
| Gene Ontology | Three ontologies (BP, MF, CC) organized as a DAG |
| GO annotations | Gene-term links with evidence codes (IDA > IEA) |
| GO enrichment | Hypergeometric test + BH FDR correction |
| KEGG pathways | Curated pathway maps with organism-specific gene sets |
| KEGG API | REST API at rest.kegg.jp for programmatic access |
| GSEA | Rank-based enrichment avoiding arbitrary gene-list thresholds |
| TM prediction | Kyte-Doolittle hydropathy + positive-inside rule |
| Workflow | Gene list -> GO + pathway enrichment -> interpretation |

## Resources

- [Gene Ontology](http://geneontology.org/) -- official GO resource
- [AmiGO 2](http://amigo.geneontology.org/) -- GO browser
- [g:Profiler](https://biit.cs.ut.ee/gprofiler/) -- multi-source enrichment analysis
- [KEGG](https://www.genome.jp/kegg/) -- pathway database
- [KEGG REST API](https://www.kegg.jp/kegg/rest/keggapi.html) -- programmatic access
- [DAVID](https://david.ncifcrf.gov/) -- functional annotation tool
- [GOATOOLS (Python)](https://github.com/tanghaibao/goatools) -- GO analysis library
- [GSEA (Broad Institute)](https://www.gsea-msigdb.org/) -- GSEA software and gene sets
- [TMHMM 2.0](https://services.healthtech.dtu.dk/services/TMHMM-2.0/) -- TM topology prediction
- [DeepTMHMM](https://biolib.com/DTU/DeepTMHMM/) -- deep learning TM prediction

---

[← Previous: Sequence Motifs and Protein Domains](../../Tier_2_Core_Bioinformatics/10_Sequence_Motifs_and_Domains/01_sequence_motifs_and_domains.ipynb) | [Next: Comparative Genomics →](../../Tier_2_Core_Bioinformatics/12_Comparative_Genomics/01_comparative_genomics.ipynb)